In [ ]:
# Define global parameters for image loading and processing
IMG_HEIGHT = 160 # A common size, can be adjusted
IMG_WIDTH = 160  # A common size, can be adjusted
BATCH_SIZE = 32

print(f"Global image dimensions set to: {IMG_HEIGHT}x{IMG_WIDTH}")

Global image dimensions set to: 160x160


In [ ]:
import os, zipfile, shutil, random
import numpy as np
import tensorflow as tf
from google.colab import files

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

raw = "/content/raw_catsdogs"
os.makedirs(raw, exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(raw)

exts = (".jpg", ".jpeg", ".png", ".bmp")
cats, dogs = [], []
for root, _, fnames in os.walk(raw):
    for f in fnames:
        if f.lower().endswith(exts):
            path = os.path.join(root, f)
            label_text = (f + root).lower()
            if "cat" in label_text:
                cats.append(path)
            elif "dog" in label_text:
                dogs.append(path)

print("Cats:", len(cats), "| Dogs:", len(dogs))

random.seed(42)
base = "/content/catsdogs"
shutil.rmtree(base, ignore_errors=True)
for label, items in [("cats", cats), ("dogs", dogs)]:
    random.shuffle(items)
    cut = int(len(items) * 0.8)
    for split, subset in [("train", items[:cut]), ("val", items[cut:])]:
        out = os.path.join(base, split, label)
        os.makedirs(out, exist_ok=True)
        for p in subset:
            shutil.copy(p, os.path.join(out, os.path.basename(p)))

train_dir = os.path.join(base, "train")
val_dir = os.path.join(base, "val")

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, seed=123, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, seed=123, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)

class_names = train_ds.class_names
print("Class names:", class_names)
print("Number of classes:", len(class_names))
print("Dataset prepared successfully!")

In [ ]:
num_classes = len(class_names)

manual_model = tf.keras.Sequential([
    tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

manual_model.summary()

In [ ]:
manual_model.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
metrics=['accuracy'])
epochs = 10 # Adjusted for better visualization of training history
history = manual_model.fit(train_ds,validation_data=val_ds,epochs=epochs)

print("Manual model training complete!")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

# Define input shape suitable for MobileNetV2, matching the dataset's image size
IMG_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 3) # Using IMG_HEIGHT and IMG_WIDTH from data loading

# Load the pre-trained MobileNetV2 model without the top classification layer
base_model = MobileNetV2(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')

# Display the base model summary
print("Base Model Summary:")
base_model.summary()

In [ ]:
# Freeze the base model to prevent its weights from being updated during training
base_model.trainable = False

print(f"Is base_model trainable? {base_model.trainable}")

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Create a new model on top of the frozen base model

# Get the output of the base model
inputs = keras.Input(shape=IMG_SHAPE)
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)

# Add a Global Average Pooling 2D layer
x = layers.GlobalAveragePooling2D()(x)

# Add a dense classification layer
num_classes = len(class_names) # Use the number of classes from the loaded dataset
outputs = layers.Dense(num_classes, activation='softmax')(x)

# Construct the new model
model = keras.Model(inputs, outputs)

# Display the new model summary
print("\nModel with Custom Classification Head Summary:")
model.summary()

In [ ]:
# Compile the model
model.compile(optimizer=keras.optimizers.Adam(),loss=keras.losses.SparseCategoricalCrossentropy(),metrics=['accuracy'])
print("Model compiled successfully!")

In [ ]:
# Train the MobileNetV2 model with the custom classification head

epochs = 10 # Adjusted for better visualization of training history
history_mobilenet = model.fit(
    train_ds,
    epochs=epochs,
    validation_data=val_ds
)

print("MobileNetV2 model training complete!")

In [ ]:
print("\n--- Evaluating Manual Model ---")
loss_manual, accuracy_manual = manual_model.evaluate(val_ds)
print(f"Manual Model - Validation Accuracy: {accuracy_manual:.4f}")

print("\n--- Evaluating MobileNetV2 Model ---")
loss_mobilenet, accuracy_mobilenet = model.evaluate(val_ds)
print(f"MobileNetV2 Model - Validation Accuracy: {accuracy_mobilenet:.4f}")

print("\n--- Comparison ---")
if accuracy_mobilenet > accuracy_manual:
    print(f"The MobileNetV2 model performed better with an accuracy of {accuracy_mobilenet:.4f} compared to the manual model's {accuracy_manual:.4f}.")
elif accuracy_manual > accuracy_mobilenet:
    print(f"The Manual Model performed better with an accuracy of {accuracy_manual:.4f} compared to the MobileNetV2 model's {accuracy_mobilenet:.4f}.")
else:
    print(f"Both models achieved the same accuracy of {accuracy_manual:.4f}.")

### Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

# Plot training history for the Manual Model
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Manual Model Training History')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot training history for the MobileNetV2 Model
plt.subplot(1, 2, 2)
plt.plot(history_mobilenet.history['accuracy'], label='Training Accuracy')
plt.plot(history_mobilenet.history['val_accuracy'], label='Validation Accuracy')
plt.title('MobileNetV2 Model Training History')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()